# 002 - Inspect Qwen Model Files

This notebook answers one mentor question:

**Where did the LoRA target module names come from?**

We will inspect the downloaded Qwen model files and find layer names like:

```text
q_proj, k_proj, v_proj, o_proj
gate_proj, up_proj, down_proj
```

These are the names we used in `LoraConfig(target_modules=[...])`.

## Big Picture

The downloaded model folder contains different kinds of files:

- `vocab.json`, `tokenizer.json`, `tokenizer_config.json`: tokenizer files. These explain how text becomes tokens.
- `config.json`: model configuration. This explains broad architecture settings.
- `model.safetensors`: model weights. This contains the actual neural network tensors and their layer names.

For LoRA, we care about **model layer names**, so we inspect `model.safetensors`.

In [10]:
from pathlib import Path

# VS Code may run notebooks from the project root or from the notebooks folder.
# This small block makes the notebook work in both cases.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL_CACHE_DIR = PROJECT_ROOT / "models" / "huggingface"
MODEL_REPO_DIR = MODEL_CACHE_DIR / "models--Qwen--Qwen2.5-Coder-0.5B-Instruct"
SNAPSHOTS_DIR = MODEL_REPO_DIR / "snapshots"

print("Project root:", PROJECT_ROOT)
print("Model repo folder exists:", MODEL_REPO_DIR.exists())
print("Snapshots folder exists:", SNAPSHOTS_DIR.exists())

Project root: d:\Code\Finetuning a model
Model repo folder exists: True
Snapshots folder exists: True


In [11]:
# Hugging Face stores downloaded models inside snapshot folders.
# The long folder name is a version/hash of the exact downloaded model snapshot.
snapshot_dirs = sorted([path for path in SNAPSHOTS_DIR.iterdir() if path.is_dir()])

for path in snapshot_dirs:
    print(path)

SNAPSHOT_DIR = snapshot_dirs[0]
print("\nUsing snapshot:", SNAPSHOT_DIR.name)

d:\Code\Finetuning a model\models\huggingface\models--Qwen--Qwen2.5-Coder-0.5B-Instruct\snapshots\ea3f2471cf1b1f0db85067f1ef93848e38e88c25

Using snapshot: ea3f2471cf1b1f0db85067f1ef93848e38e88c25


In [12]:
# These are the important files for this lesson.
config_file = SNAPSHOT_DIR / "config.json"
tokenizer_file = SNAPSHOT_DIR / "tokenizer.json"
vocab_file = SNAPSHOT_DIR / "vocab.json"
weights_file = SNAPSHOT_DIR / "model.safetensors"

for file_path in [config_file, tokenizer_file, vocab_file, weights_file]:
    print(file_path.name, "exists:", file_path.exists(), "size MB:", round(file_path.stat().st_size / 1024 / 1024, 2))

config.json exists: True size MB: 0.0
tokenizer.json exists: True size MB: 6.71
vocab.json exists: True size MB: 2.65
model.safetensors exists: True size MB: 942.32


## Read the Model Config

`config.json` is human-readable. It does not list every weight tensor, but it tells us what kind of model this is and gives architecture-level details.

In [13]:
import json

with config_file.open("r", encoding="utf-8") as f:
    config = json.load(f)

important_config_keys = [
    "architectures",
    "model_type",
    "num_hidden_layers",
    "hidden_size",
    "num_attention_heads",
    "num_key_value_heads",
    "vocab_size",
]

for key in important_config_keys:
    print(f"{key}:", config.get(key))

architectures: ['Qwen2ForCausalLM']
model_type: qwen2
num_hidden_layers: 24
hidden_size: 896
num_attention_heads: 14
num_key_value_heads: 2
vocab_size: 151936


## Read the Weight Names from `model.safetensors`

`model.safetensors` is binary, so we do not open it like a text file. We use the `safetensors` library.

Good news: we can list tensor names without loading the full model into GPU memory.

In [14]:
from safetensors import safe_open

with safe_open(weights_file, framework="pt", device="cpu") as f:
    tensor_names = list(f.keys())

print("Number of tensors:", len(tensor_names))
print("\nFirst 20 tensor names:")
for name in tensor_names[:20]:
    print(name)

Number of tensors: 290

First 20 tensor names:
model.embed_tokens.weight
model.layers.0.input_layernorm.weight
model.layers.0.mlp.down_proj.weight
model.layers.0.mlp.gate_proj.weight
model.layers.0.mlp.up_proj.weight
model.layers.0.post_attention_layernorm.weight
model.layers.0.self_attn.k_proj.bias
model.layers.0.self_attn.k_proj.weight
model.layers.0.self_attn.o_proj.weight
model.layers.0.self_attn.q_proj.bias
model.layers.0.self_attn.q_proj.weight
model.layers.0.self_attn.v_proj.bias
model.layers.0.self_attn.v_proj.weight
model.layers.1.input_layernorm.weight
model.layers.1.mlp.down_proj.weight
model.layers.1.mlp.gate_proj.weight
model.layers.1.mlp.up_proj.weight
model.layers.1.post_attention_layernorm.weight
model.layers.1.self_attn.k_proj.bias
model.layers.1.self_attn.k_proj.weight


## Find the LoRA Target Modules

Now we search for the exact names we used in `target_modules`.

The full tensor names look like this:

```text
model.layers.0.self_attn.q_proj.weight
```

But LoRA only needs the short module name:

```text
q_proj
```

In [15]:
target_modules = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

matching_names = [
    name
    for name in tensor_names
    if any(module_name in name for module_name in target_modules)
]

print("Matching tensor names:", len(matching_names))
print("\nFirst 40 matches:")
for name in matching_names[:40]:
    print(name)

Matching tensor names: 240

First 40 matches:
model.layers.0.mlp.down_proj.weight
model.layers.0.mlp.gate_proj.weight
model.layers.0.mlp.up_proj.weight
model.layers.0.self_attn.k_proj.bias
model.layers.0.self_attn.k_proj.weight
model.layers.0.self_attn.o_proj.weight
model.layers.0.self_attn.q_proj.bias
model.layers.0.self_attn.q_proj.weight
model.layers.0.self_attn.v_proj.bias
model.layers.0.self_attn.v_proj.weight
model.layers.1.mlp.down_proj.weight
model.layers.1.mlp.gate_proj.weight
model.layers.1.mlp.up_proj.weight
model.layers.1.self_attn.k_proj.bias
model.layers.1.self_attn.k_proj.weight
model.layers.1.self_attn.o_proj.weight
model.layers.1.self_attn.q_proj.bias
model.layers.1.self_attn.q_proj.weight
model.layers.1.self_attn.v_proj.bias
model.layers.1.self_attn.v_proj.weight
model.layers.10.mlp.down_proj.weight
model.layers.10.mlp.gate_proj.weight
model.layers.10.mlp.up_proj.weight
model.layers.10.self_attn.k_proj.bias
model.layers.10.self_attn.k_proj.weight
model.layers.10.self_

## Group by Module Type

This helps us see that the same module names repeat across many transformer layers.

In [16]:
from collections import Counter

counts = Counter()

for name in tensor_names:
    for module_name in target_modules:
        if f".{module_name}." in name:
            counts[module_name] += 1

for module_name in target_modules:
    print(f"{module_name}: {counts[module_name]} tensors")

q_proj: 48 tensors
k_proj: 48 tensors
v_proj: 48 tensors
o_proj: 24 tensors
gate_proj: 24 tensors
up_proj: 24 tensors
down_proj: 24 tensors


## Inspect Tensor Shapes

A tensor shape tells us the size of a weight matrix. We do not need to deeply understand the math yet. For now, notice that these are large matrices, which is why full fine-tuning is expensive.

In [17]:
examples_to_inspect = [
    "model.layers.0.self_attn.q_proj.weight",
    "model.layers.0.self_attn.k_proj.weight",
    "model.layers.0.self_attn.v_proj.weight",
    "model.layers.0.self_attn.o_proj.weight",
    "model.layers.0.mlp.gate_proj.weight",
    "model.layers.0.mlp.up_proj.weight",
    "model.layers.0.mlp.down_proj.weight",
]

with safe_open(weights_file, framework="pt", device="cpu") as f:
    for name in examples_to_inspect:
        tensor = f.get_tensor(name)
        print(name, tuple(tensor.shape))

model.layers.0.self_attn.q_proj.weight (896, 896)
model.layers.0.self_attn.k_proj.weight (128, 896)
model.layers.0.self_attn.v_proj.weight (128, 896)
model.layers.0.self_attn.o_proj.weight (896, 896)
model.layers.0.mlp.gate_proj.weight (4864, 896)
model.layers.0.mlp.up_proj.weight (4864, 896)
model.layers.0.mlp.down_proj.weight (896, 4864)


## The LoRA Config We Use

Now the LoRA target list should feel less mysterious. We chose names that really exist inside Qwen's model weights.

In [18]:
target_modules

['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']

## Lesson

We choose LoRA target modules by inspecting the model's repeated linear layers.

- `q_proj`, `k_proj`, `v_proj`, `o_proj` are attention projection layers.
- `gate_proj`, `up_proj`, `down_proj` are feed-forward / MLP projection layers.

For this text-to-SQL project, that is a strong starting point because the model needs to learn both:

1. How to pay attention to the right schema/table/column names.
2. How to generate better SQL structure from that information.